In [2]:
import numpy as np

In [7]:
def get_disp(coords, latt):
   """
   Calculate displacements for NVT trajectories.

   :param coords: Fractional coordinates for all atoms.
   :param latt: Lattice descriptions.

   :return: Numpy array of with shape [site, time step, axis] describing displacements.
   """
   coords = np.concatenate(coords, axis=1)
   d_coords = coords[:, 1:] - coords[:, :-1]
   d_coords = d_coords - np.round(d_coords)
   f_disp = np.cumsum(d_coords, axis=1)
   latt = np.array(latt)
   print(f_disp.shape, latt.shape)
   disp = np.einsum('ijk,jkl->ijk', f_disp, latt[1:])
   return disp


def get_disp_npt(coords, latt):
   """
   Calculate displacements for NPT trajectories.

   :param coords: Fractional coordinates for all atoms.
   :param latt: Lattice descriptions.

   :return: Numpy array with shape [site, timestep, axis] describing displacements.
   """
   coords = np.concatenate(coords, axis=1)
   wrapped = np.einsum('ijk,jkl->ijk', coords, latt)
   wrapped_diff = np.diff(wrapped, axis=1)
   latt_para = np.einsum('ijj->ij', latt)

   unwrapped_disp = wrapped_diff - np.floor(wrapped_diff / latt_para[1:] + 1 / 2) * latt_para[1:]

   return unwrapped_disp


def get_disp_npt_matrix(coords, latt):
   """
   Calculate displacements for NPT trajectories.

   :param coords: Fractional coordinates for all atoms.
   :param latt: Lattice descriptions.

   :return: Numpy array with shape [site, timestep, axis] describing displacements.
   """
   coords = np.concatenate(coords, axis=1)
   latt_inv = np.linalg.inv(latt)
   wrapped = np.einsum('ijk,jkl->ijk', coords, latt)
   wrapped_diff = np.diff(wrapped, axis=1)

   unwrapped_disp = wrapped_diff - np.einsum(
      'ijk,jkl->ijk', np.floor(np.einsum('ijk,jkl->ijk', wrapped_diff, latt_inv[1:]) + (1 / 2)), latt[1:])

   return unwrapped_disp

In [20]:
coords = np.array([[[0.1,0.1,0.1],[0.1,0.1,0.1],[0.1,0.1,0.1],[0.1,0.1,0.1]],
                   [[0.1,0.1,0.1],[0.1,0.1,0.1],[0.1,0.1,0.1],[0.1,0.1,0.1]],
                   [[0.2,0.1,0.1],[0.2,0.1,0.1],[0.2,0.1,0.1],[0.2,0.1,0.1]],
                   [[0.3,0.1,0.1],[0.3,0.1,0.1],[0.3,0.1,0.1],[0.3,0.1,0.1]],
                   [[0.4,0.1,0.1],[0.4,0.1,0.1],[0.4,0.1,0.1],[0.4,0.1,0.1]]
                   ])
coords = np.expand_dims(coords, 2)

latt = np.array([[[1,0,0],[0,1,0],[0,0,1]],
                 [[1,0,0],[0,1,0],[0,0,1]],
                 [[1,0,0],[0,1,0],[0,0,1]],
                 [[1,0,0],[0,1,0],[0,0,1]],
                 [[1,0,0],[0,1,0],[0,0,1]],
                ])

In [21]:
print(get_disp(coords, latt))

(4, 4, 3) (5, 3, 3)
[[[0.  0.  0. ]
  [0.1 0.  0. ]
  [0.2 0.  0. ]
  [0.3 0.  0. ]]

 [[0.  0.  0. ]
  [0.1 0.  0. ]
  [0.2 0.  0. ]
  [0.3 0.  0. ]]

 [[0.  0.  0. ]
  [0.1 0.  0. ]
  [0.2 0.  0. ]
  [0.3 0.  0. ]]

 [[0.  0.  0. ]
  [0.1 0.  0. ]
  [0.2 0.  0. ]
  [0.3 0.  0. ]]]


In [18]:
print(get_disp_npt(coords, latt))

[[[0.   0.   0.  ]
  [0.12 0.   0.  ]
  [0.11 0.   0.  ]
  [0.11 0.   0.  ]]

 [[0.   0.   0.  ]
  [0.12 0.   0.  ]
  [0.11 0.   0.  ]
  [0.11 0.   0.  ]]

 [[0.   0.   0.  ]
  [0.12 0.   0.  ]
  [0.11 0.   0.  ]
  [0.11 0.   0.  ]]

 [[0.   0.   0.  ]
  [0.12 0.   0.  ]
  [0.11 0.   0.  ]
  [0.11 0.   0.  ]]]


In [19]:
print(get_disp_npt_matrix(coords, latt))

[[[0.   0.   0.  ]
  [0.12 0.   0.  ]
  [0.11 0.   0.  ]
  [0.11 0.   0.  ]]

 [[0.   0.   0.  ]
  [0.12 0.   0.  ]
  [0.11 0.   0.  ]
  [0.11 0.   0.  ]]

 [[0.   0.   0.  ]
  [0.12 0.   0.  ]
  [0.11 0.   0.  ]
  [0.11 0.   0.  ]]

 [[0.   0.   0.  ]
  [0.12 0.   0.  ]
  [0.11 0.   0.  ]
  [0.11 0.   0.  ]]]
